In [27]:
;; Base do Grafo

(define graph-schema
  '((directed . 0) (weighted . 1) (direct-loop . 2) (body . 3)))

(define (getp g p)
  (list-ref g (cdr (assq p graph-schema))))

(define (replace-by-i i value l)
  (if (= i 0)
      (cons value (cdr l))
      (cons (car l) (replace-by-i (- i 1) value (cdr l)))))

(define (setp g p value)
  (replace-by-i (cdr (assq p graph-schema)) value g))


;; Função Auxiliar

; Remove todas as ocorrências de um elemento 'x' de uma lista simples 'lst'
(define (remove-elemento x lst)
  (cond
    ((null? lst) '())
    ((eq? x (car lst)) (remove-elemento x (cdr lst)))
    (else (cons (car lst) (remove-elemento x (cdr lst))))))


;; Lógica de remoção de arestas e vértices

; Recebe o corpo do grafo e retorna um novo corpo sem o vértice 'v'
(define (remove-vertice-do-corpo corpo v)
  (cond
    ((null? corpo) '())
    (else
     (let ((linha-atual (car corpo)))
       (let ((vertice-dono (car linha-atual))
             (conexoes (cdr linha-atual)))
         
         (if (eq? vertice-dono v)
             (remove-vertice-do-corpo (cdr corpo) v)
             (cons (cons vertice-dono (remove-elemento v conexoes))
                   (remove-vertice-do-corpo (cdr corpo) v))))))))

; Recebe o corpo do grafo e retorna um novo corpo sem a aresta
(define (remove-aresta-do-corpo corpo aresta direcionado?)
  (let ((u (car aresta))   
        (v (cadr aresta))) 
    
    (define (processa-corpo c)
      (cond
        ((null? c) '())
        (else
         (let ((linha (car c)))
           (let ((vertice-dono (car linha))
                 (conexoes (cdr linha)))
             
             (cond
               ((eq? vertice-dono u)
                (cons (cons vertice-dono (remove-elemento v conexoes))
                      (processa-corpo (cdr c))))
               
               ((and (not direcionado?) (eq? vertice-dono v))
                (cons (cons vertice-dono (remove-elemento u conexoes))
                      (processa-corpo (cdr c))))
               
               (else
                (cons linha (processa-corpo (cdr c))))))))))
    
    (processa-corpo corpo)))


;; Encapsulamento

(define (remove-vertice g v)
  (setp g 'body (remove-vertice-do-corpo (getp g 'body) v)))

(define (remove-aresta g aresta)
  (setp g 'body (remove-aresta-do-corpo (getp g 'body) aresta (getp g 'directed))))


;; A Primitiva

; Função principal que o usuário vai chamar.
(define (remove-do-grafo g . elementos)
  ; Função auxiliar com recursão de cauda para processar a lista de elementos
  (define (processa grafo-atual elems)
    (if (null? elems)
        grafo-atual
        (let ((item (car elems))
              (resto (cdr elems)))
          (cond
            ((symbol? item) 
             (processa (remove-vertice grafo-atual item) resto))
            
            ((pair? item)   
             (processa (remove-aresta grafo-atual item) resto))
            
            (else           
             (processa grafo-atual resto))))))
  
  (processa g elementos))